In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             roc_auc_score)
import os


In [2]:
load_dotenv()
db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")


# Engine connect with DB
engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

# Verificar que la conexión funciona
with engine.connect() as conn:
    print("Conexión exitosa")

Conexión exitosa


In [3]:
df_rfm = pd.read_sql("SELECT * FROM rfm_final", engine)

#Check dataframe
print(f"Rows: {df_rfm.shape[0]:,}")
print(f"Cols: {df_rfm.shape[1]}")
print("-----------------------------")

#Check data types
print(df_rfm.dtypes)
print("-----------------------------")

#nulls
print(df_rfm.isnull().sum())
#Segments
print(df_rfm.segment.unique())

df_rfm

Rows: 93,358
Cols: 11
-----------------------------
customer_unique_id     object
recency_days            int64
frequency               int64
monetary_value        float64
avg_review_score      float64
r_score                 int64
f_score                 int64
m_score                 int64
experience_score        int64
rfm_total               int64
segment                object
dtype: object
-----------------------------
customer_unique_id      0
recency_days            0
frequency               0
monetary_value          0
avg_review_score      603
r_score                 0
f_score                 0
m_score                 0
experience_score        0
rfm_total               0
segment                 0
dtype: int64
['Promising' 'New customer' 'Need attention' 'Champions'
 'Potential loyalist' 'Loyal' 'About to sleep' 'Cannot lose' 'Hibernating'
 'At risk' 'Lost']


,customer_unique_id,recency_days,frequency,monetary_value,avg_review_score,r_score,f_score,m_score,experience_score,rfm_total,segment
0,7a22d14aa3c3599238509ddca4b93b01,49,1,63.90,1.0,5,0,2,1,8,Promising
1,21dbe8eabd00b34492a939c540e2b1a7,49,1,6.90,5.0,5,0,1,3,9,New customer
2,b701bebbdf478f5500348f03aff62121,49,1,24.90,3.0,5,0,1,1,7,New customer
3,7febafa06d9d8f232a900a2937f04338,49,1,45.90,5.0,5,0,2,5,12,Promising
4,f80013faf776e37bcea7634d59c2181e,49,1,65.00,5.0,5,0,2,4,11,Promising
...,...,...,...,...,...,...,...,...,...,...,...
93353,7390ed59fa1febbfda31a80b4318c8cb,744,1,128.90,5.0,1,0,4,5,10,At risk
93354,87776adb449c551e74c13fc34f036105,744,1,29.99,5.0,1,0,1,5,7,Lost
93355,61db744d2f835035a5625b59350c6b63,744,1,36.49,3.0,1,0,1,1,3,Lost
93356,2f64e403852e6893ae37485d5fcacdaf,744,1,21.90,4.0,1,0,1,2,4,Lost


In [4]:
#A list is defined based on RFM, segments with a probability of churn - This is a hypotetical business rule
#churn = 1 if customers is in list
#churn = 0 if customer is not in list

churn_segment = ['Lost','Hibernating','At risk','Cannot lose']


df_rfm['churn'] = df_rfm['segment'].isin(churn_segment).astype(int)

print(df_rfm.churn.value_counts())
print("----------------------------------------------------------")

print(df_rfm.groupby('churn')['segment'].value_counts())

churn
0    63483
1    29875
Name: count, dtype: int64
----------------------------------------------------------
churn  segment           
0      Need attention        26807
       Promising             14418
       New customer           7373
       About to sleep         7363
       Potential loyalist     5859
       Champions               938
       Loyal                   725
1      Hibernating           11440
       Cannot lose            7519
       At risk                6989
       Lost                   3927
Name: count, dtype: int64


In [5]:
df_rfm.columns

Index(['customer_unique_id', 'recency_days', 'frequency', 'monetary_value',
       'avg_review_score', 'r_score', 'f_score', 'm_score', 'experience_score',
       'rfm_total', 'segment', 'churn'],
      dtype='object')

In [6]:
#We start to define the model
FEATURES = ['recency_days',
            'frequency',
            'monetary_value',
            'avg_review_score',
            'r_score',
            'f_score',
            'm_score',
            'experience_score'
            ]

TARGET = 'churn'
#Data imputation with nulls in avg_review_score
#Select the data imputation with median because is strong with outliers
df_rfm['avg_review_score'] = df_rfm['avg_review_score'].fillna(df_rfm['avg_review_score'].median())

#Define X and Y

X = df_rfm[FEATURES]
y = df_rfm[TARGET]

#Separe Split the data: 80% Train. 20% test
#Stratify = guarantee same proportion with churn
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=41,
    stratify=y
)

#Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} filas")
print(f"Test: {X_test.shape[0]:,} filas")
print(f"Churn in train: {y_train.mean():.1%} filas")
print(f"Churn in test: {y_test.mean():.1%} filas")


Train: 74,686 filas
Test: 18,672 filas
Churn in train: 32.0% filas
Churn in test: 32.0% filas


In [7]:
#First model - logistic regression
lr = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_scaled,y_train)

#Predictions
y_pred = lr.predict(X_test)
y_pred_prob = lr.predict_proba(X_test_scaled)[:,1]

#Evaluation
print("---clasification report---")
print(classification_report(y_test,y_pred,
                            target_names=['No churn','Churn']))
print("---AUC ROC---")
print(f'AUC ROC: {roc_auc_score(y_test,y_pred_prob):.4f}')

print("--- Confusion matrix ---")
print(confusion_matrix(y_test,y_pred))

---clasification report---
              precision    recall  f1-score   support

    No churn       0.68      0.97      0.80     12697
       Churn       0.11      0.01      0.02      5975

    accuracy                           0.66     18672
   macro avg       0.40      0.49      0.41     18672
weighted avg       0.50      0.66      0.55     18672

---AUC ROC---
AUC ROC: 0.9459
--- Confusion matrix ---
[[12312   385]
 [ 5925    50]]


c:\Users\carlos.yepes\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [8]:
#Find the optimal thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)

#F1 for each umbral
f1_scores = 2*(precisions*recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_thresholds = thresholds[best_idx]

print(f'threshold optimal: {best_thresholds:.4f}')
print(f'precision in threshold: {precisions[best_idx]:.4f}')
print(f'recall in threshold: {recalls[best_idx]:.4f}')
print(f'F1 in threshold: {f1_scores[best_idx]:.4f}')

print()

y_pred_optimal = (y_pred >= best_thresholds).astype(int)

print('--- Classification report with optimal threshold ---')
print(classification_report(y_test,y_pred_optimal,
                            target_names=['No churn','Churn']))

threshold optimal: 0.5331
precision in threshold: 0.7969
recall in threshold: 1.0000
F1 in threshold: 0.8870

--- Classification report with optimal threshold ---
              precision    recall  f1-score   support

    No churn       0.68      0.97      0.80     12697
       Churn       0.11      0.01      0.02      5975

    accuracy                           0.66     18672
   macro avg       0.40      0.49      0.41     18672
weighted avg       0.50      0.66      0.55     18672



In [9]:
#threshold analysis

threshold_s = np.arange(0.1, 0.9 ,0.05)
results = []

for threshold in threshold_s:
    y_pred_tmp = (y_pred_prob >=threshold).astype(int)

    tp = ((y_pred_tmp ==1) & (y_test ==1)).sum()
    fp = ((y_pred_tmp ==1) & (y_test ==0)).sum()
    fn = ((y_pred_tmp ==0) & (y_test ==1)).sum()

    precision = tp/ (tp+fp) if (tp+fp) > 0 else 0
    recall = tp/ (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2*precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    results.append({
        'threshold': round(threshold,2),
        'precision': round(precision,3),
        'recall' : round(recall,3),
        'f1': round(f1,3),
        'churn detected': tp
    })

df_threshold = pd.DataFrame(results)
print(df_threshold.to_string(index=False))

 threshold  precision  recall    f1  churn detected
      0.10      0.595   1.000 0.746            5975
      0.15      0.725   1.000 0.841            5975
      0.20      0.785   1.000 0.880            5975
      0.25      0.790   1.000 0.883            5975
      0.30      0.793   1.000 0.884            5975
      0.35      0.794   1.000 0.885            5975
      0.40      0.796   1.000 0.886            5975
      0.45      0.796   1.000 0.886            5975
      0.50      0.797   1.000 0.887            5975
      0.55      0.797   0.999 0.887            5972
      0.60      0.798   0.987 0.883            5897
      0.65      0.802   0.946 0.868            5651
      0.70      0.803   0.872 0.836            5210
      0.75      0.809   0.791 0.800            4724
      0.80      0.819   0.668 0.736            3991
      0.85      0.816   0.532 0.644            3177


In [10]:
#Probabilistic distributions

print("Stadistic about y_pred_proba:")
print(pd.Series(y_pred_prob).describe())

print("----------------------------------------------------------")

print(f"prob < 0.10: {(y_pred_prob < 0.10).sum():,}")
print(f"prob < 0.30: {(y_pred_prob < 0.30).sum():,}")
print(f"prob < 0.50: {(y_pred_prob < 0.50).sum():,}")
print(f"prob >= 0.50: {(y_pred_prob >= 0.50).sum():,}")

Stadistic about y_pred_proba:
count    18672.000000
mean         0.371049
std          0.410423
min          0.000091
25%          0.004140
50%          0.118479
75%          0.807921
max          0.999998
dtype: float64
----------------------------------------------------------
prob < 0.10: 8,632
prob < 0.30: 11,136
prob < 0.50: 11,174
prob >= 0.50: 7,498


In [11]:
# Model 2: Without scores RFM — variables brutas

GROSS_FEATURS = [
    'recency_days',
    'frequency',
    'monetary_value',
    'avg_review_score'
]

X2       = df_rfm[GROSS_FEATURS]
y2       = df_rfm[TARGET]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=41,
    stratify=y2
)

scaler2       = StandardScaler()
X2_train_sc   = scaler2.fit_transform(X2_train)
X2_test_sc    = scaler2.transform(X2_test)

lr2 = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

lr2.fit(X2_train_sc, y2_train)

y2_pred       = lr2.predict(X2_test_sc)
y2_pred_proba = lr2.predict_proba(X2_test_sc)[:, 1]

print("=== Classification Report — features brutas ===")
print(classification_report(y2_test, y2_pred,
                             target_names=['No churn', 'Churn']))
print(f"AUC-ROC: {roc_auc_score(y2_test, y2_pred_proba):.4f}")
print()

print("Estadísticas de probabilidades:")
print(pd.Series(y2_pred_proba).describe())

=== Classification Report — features brutas ===
              precision    recall  f1-score   support

    No churn       0.95      0.90      0.92     12697
       Churn       0.80      0.89      0.84      5975

    accuracy                           0.89     18672
   macro avg       0.87      0.89      0.88     18672
weighted avg       0.90      0.89      0.90     18672

AUC-ROC: 0.9433

Estadísticas de probabilidades:
count    18672.000000
mean         0.386757
std          0.377080
min          0.003187
25%          0.034620
50%          0.229875
75%          0.791513
max          0.999853
dtype: float64


In [ ]:
#Logistic regression

coeficientes = pd.DataFrame({
    'feature':     GROSS_FEATURS,
    'coeficiente': lr2.coef_[0]
}).sort_values('coeficiente', ascending=False)

print(coeficientes.to_string(index=False))

         feature  coeficiente
    recency_days     3.090888
  monetary_value     0.196114
       frequency     0.009356
avg_review_score     0.001391


In [ ]:
# Model 3: Random Forest

rf = RandomForestClassifier(
    n_estimators=100,      # 100 tree
    class_weight='balanced',
    random_state=42,
    n_jobs=-1              
)

rf.fit(X2_train, y2_train)

y_rf_pred       = rf.predict(X2_test)
y_rf_pred_proba = rf.predict_proba(X2_test)[:, 1]

print("=== Classification Report — Random Forest ===")
print(classification_report(y2_test, y_rf_pred,
                             target_names=['No churn', 'Churn']))
print(f"AUC-ROC: {roc_auc_score(y2_test, y_rf_pred_proba):.4f}")

importancias = pd.DataFrame({
    'feature':    GROSS_FEATURS,
    'importancia': rf.feature_importances_
}).sort_values('importancia', ascending=False)

print("\n=== importance de features ===")
print(importancias.to_string(index=False))

=== Classification Report — Random Forest ===
              precision    recall  f1-score   support

    No churn       1.00      1.00      1.00     12697
       Churn       1.00      1.00      1.00      5975

    accuracy                           1.00     18672
   macro avg       1.00      1.00      1.00     18672
weighted avg       1.00      1.00      1.00     18672

AUC-ROC: 0.9997

=== Importancia de features ===
         feature  importancia
    recency_days     0.794067
  monetary_value     0.204569
avg_review_score     0.001054
       frequency     0.000310


In [ ]:
# Model 4: Controled Random Forest

rf2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,           # Mx 6 levels per tree
    min_samples_leaf=50,   # each leaf needs at least 50 customers
    max_features='sqrt',   # each tree sees roots(4) = 2 features
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf2.fit(X2_train, y2_train)

y_rf2_pred       = rf2.predict(X2_test)
y_rf2_pred_proba = rf2.predict_proba(X2_test)[:, 1]

print("=== Classification Report — Random Forest controlado ===")
print(classification_report(y2_test, y_rf2_pred,
                             target_names=['No churn', 'Churn']))
print(f"AUC-ROC: {roc_auc_score(y2_test, y_rf2_pred_proba):.4f}")

print("\n=== Importancia de features ===")
importancias2 = pd.DataFrame({
    'feature':     GROSS_FEATURS,
    'importancia': rf2.feature_importances_
}).sort_values('importancia', ascending=False)
print(importancias2.to_string(index=False))

=== Classification Report — Random Forest controlado ===
              precision    recall  f1-score   support

    No churn       1.00      1.00      1.00     12697
       Churn       1.00      1.00      1.00      5975

    accuracy                           1.00     18672
   macro avg       1.00      1.00      1.00     18672
weighted avg       1.00      1.00      1.00     18672

AUC-ROC: 1.0000

=== Importancia de features ===
         feature  importancia
    recency_days     0.821293
  monetary_value     0.178121
avg_review_score     0.000342
       frequency     0.000244


In [ ]:
# Diagnostic

print("=== Distribución de recency_days por churn ===")
print(df_rfm.groupby('churn')['recency_days'].describe().round(1))

print("\n=== Distribución de monetary_value por churn ===")
print(df_rfm.groupby('churn')['monetary_value'].describe().round(1))

print("\n=== Punto de corte natural en recency_days ===")
# ¿Existe un umbral simple que separe perfectamente churn de no churn?
for dias in [200, 250, 300, 350, 400]:
    churn_arriba  = df_rfm[df_rfm['recency_days'] >= dias]['churn'].mean()
    churn_abajo   = df_rfm[df_rfm['recency_days'] <  dias]['churn'].mean()
    print(f"recency >= {dias}: {churn_arriba:.1%} churn | "
          f"recency < {dias}: {churn_abajo:.1%} churn")

=== Distribución de recency_days por churn ===
         count   mean    std    min    25%    50%    75%    max
churn                                                          
0      63483.0  212.6  114.1   49.0  126.0  200.0  272.0  744.0
1      29875.0  443.6   94.0  317.0  361.0  431.0  515.0  762.0

=== Distribución de monetary_value por churn ===
         count   mean    std  min   25%   50%    75%      max
churn                                                        
0      63483.0  135.5  195.9  0.8  50.0  89.9  145.0   7160.0
1      29875.0  156.6  255.3  2.3  39.9  67.6  180.0  13440.0

=== Punto de corte natural en recency_days ===
recency >= 200: 48.4% churn | recency < 200: 0.0% churn
recency >= 250: 59.1% churn | recency < 250: 0.0% churn
recency >= 300: 73.5% churn | recency < 300: 0.0% churn
recency >= 350: 80.0% churn | recency < 350: 9.9% churn
recency >= 400: 80.2% churn | recency < 400: 16.5% churn


In [ ]:
# Churn = Customer who bought exactly one time
# Business problem changes to:
# ¿Can we predict if
# the customer will be back?

df_rfm['churn_v2'] = (df_rfm['frequency'] == 1).astype(int)

print(df_rfm['churn_v2'].value_counts())
print(f"\nChurn rate: {df_rfm['churn_v2'].mean():.1%}")

churn_v2
1    90557
0     2801
Name: count, dtype: int64

Churn rate: 97.0%


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#  ROC
RocCurveDisplay.from_predictions(
    y2_test, y2_pred_proba,
    ax=axes[0],
    name='Regresión Logística'
)
axes[0].set_title('Curva ROC — Modelo de Churn')
axes[0].plot([0,1], [0,1], 'k--', label='Random (AUC = 0.50)')
axes[0].legend()

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y2_test, y2_pred,
    display_labels=['No churn', 'Churn'],
    cmap='Blues',
    ax=axes[1]
)
axes[1].set_title('Matriz de Confusión')

plt.tight_layout()
plt.savefig('churn_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphic saved as churn_model_evaluation.png")

Gráfico guardado como churn_model_evaluation.png


C:\Users\carlos.yepes\AppData\Local\Temp\ipykernel_23196\1877324523.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
